In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-transpiler
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Transpiler Serviceを使ってリモートで回路をトランスパイルする

> **Danger:** 2025年7月18日以降、このサービスは新しいIBM Quantum&reg; Platformをサポートするための移行中であり、現在は利用できません。AIパスについては、[ローカルモード](/guides/ai-transpiler-passes#run-the-ai-transpiler-passes-locally-or-on-the-cloud)をご使用ください。
> 
>     このサービスはベータリリースであり、変更される可能性があります。
>     フィードバックや開発チームへのお問い合わせは、[Qiskit Slack Workspace](https://qiskit.slack.com/archives/C06KF8YHUAU) チャンネルをご利用ください。

Qiskit Transpiler Serviceは、クラウド上でトランスパイル機能を提供します。ローカルのQiskitトランスパイラーの機能に加えて、IBM QuantumクラウドリソースおよびAIを活用したトランスパイラーパスを使用することができます。

Qiskit Transpiler Serviceは、このサービスとその機能を既存のQiskitパターンおよびワークフローにシームレスに統合するためのPythonライブラリを提供しています。このサービスは、IBM Quantum Premium Plan、Flex Plan、およびOn-Prem（IBM Quantum Platform API経由）Planのユーザーのみご利用いただけます。

<span id="install-transpiler-service"></span>
## qiskit-ibm-transpilerパッケージのインストール
Qiskit Transpiler Serviceを使用するには、`qiskit-ibm-transpiler` パッケージをインストールしてください。

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="false",
    optimization_level=3,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

このパッケージは、[IBM Quantum Platformの認証情報](/guides/cloud-setup)を使用して自動的に認証します。認証方法は、[Qiskit Runtimeの管理方法](/guides/initialize-account)に準拠しています。
- 環境変数: `QISKIT_IBM_TOKEN`
- 設定ファイル `~/.qiskit/qiskit-ibm.json`（`default-ibm-quantum` セクション配下）

*注意*: このパッケージはQiskit SDK v1.Xが必要です。

## qiskit-ibm-transpilerのトランスパイルオプション
- `backend_name`（オプション、str）- QiskitRuntimeServiceで指定する形式のバックエンド名（例: `ibm_torino`）。これを設定すると、トランスパイルメソッドは指定されたバックエンドのレイアウトをトランスパイル処理に使用します。`coupling_map` など、この設定に影響を与える他のオプションが設定された場合、`backend_name` の設定は上書きされます。
- `coupling_map`（オプション、List[List[int]]）- 有効なカップリングマップのリスト（例: [[0,1],[1,2]]）。これを設定すると、トランスパイルメソッドはこのカップリングマップをトランスパイル処理に使用します。定義された場合、`target` に指定された値を上書きします。
- `optimization_level`（int）- トランスパイル処理中に適用される最適化レベルです。有効な値は [1,2,3] で、1が最も低い最適化（最速）、3が最も高い最適化（最も時間がかかります）です。
- `ai`（"true"、"false"、"auto"）- トランスパイル中にAIを活用した機能を使用するかどうかを指定します。利用可能なAI機能には、`AIRouting` トランスパイルパスやその他のAIを活用した合成メソッドが含まれます。`"true"` の場合、サービスは指定された `optimization_level` に応じてさまざまなAI対応トランスパイルパスを適用します。`"false"` の場合は、AIなしで最新のQiskitトランスパイル機能を使用します。`"auto"` の場合は、回路に基づいて標準のQiskitヒューリスティックパスまたはAI対応パスを適用するかどうかをサービスが判断します。
- `qiskit_transpile_options`（dict）- [Qiskitの `transpile()` メソッド](defaults-and-configuration-options)で有効なその他のオプションを含むことができるPython辞書オブジェクトです。`qiskit_transpile_options` の入力に `optimization_level` が含まれている場合、パラメーター入力として指定された `optimization_level` が優先されます。`qiskit_transpile_options` にQiskitの `transpile()` メソッドで認識されないオプションが含まれている場合、ライブラリはエラーを発生させます。

利用可能な `qiskit-ibm-transpiler` メソッドの詳細については、[qiskit-ibm-transpiler APIリファレンス](https://docs.quantum.ibm.com/api/qiskit-ibm-transpiler)をご覧ください。サービスAPIの詳細については、[Qiskit Transpiler Service REST APIドキュメント](https://docs.quantum.ibm.com/api/qiskit-transpiler-service-rest)をご覧ください。

## 使用例
以下の例では、さまざまなパラメーターを使用してQiskit Transpiler Serviceで回路をトランスパイルする方法を示します。

1. 回路を作成し、`backend_name` に `ibm_torino`、`optimization_level` に 3 を指定し、トランスパイル中にAIを使用せずにQiskit Transpiler Serviceを呼び出します。

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="true",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

*注意*: `backend_name` で指定できるのは、お使いのIBM Quantumアカウントでアクセスできるデバイスのみです。`backend_name` の他に、`TranspilerService` では `coupling_map` もパラメーターとして使用できます。

2. 同様の回路を生成し、`ai` フラグを `True` に設定してAIトランスパイル機能を要求した上でトランスパイルします。

In [ ]:
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_transpiler.transpiler_service import TranspilerService

circuit = efficient_su2(101, entanglement="circular", reps=1)

cloud_transpiler_service = TranspilerService(
    backend_name="ibm_torino",
    ai="auto",
    optimization_level=1,
)
transpiled_circuit = cloud_transpiler_service.run(circuit)

3. 同様の回路を生成し、AIを活用したトランスパイルパスを使用するかどうかをサービスに判断させながらトランスパイルします。